# MapReduce Analytics (M1–M5), Validation & Performance
This notebook parses and presents the YARN MapReduce outputs (`m1_results.txt` through `m5_results.txt`), compares 3-month LTA Bus OD trends across August, September, and October 2026, validates MapReduce outputs against independent Pandas checks, documents the self-generated analyses (M4 & M5) with logic diagrams, and analyses Task 3 YARN benchmark performance (`benchmark_summary.csv`).

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("userfiles") if Path("userfiles").exists() else Path("..") / "userfiles"
if not BASE_DIR.exists():
    BASE_DIR = Path(".")

RESULTS_DIR = BASE_DIR / "mapreduce_results"
BENCH_DIR = BASE_DIR / "benchmarks"
CLEAN_DIR = BASE_DIR / "processed_data"


# Parse tab-delimited mrjob output lines into (key_string, payload_dict) tuples
def parse_mrjob_output(file_path):
    records = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or "\t" not in line:
                continue
            key_raw, val_raw = line.split("\t", 1)
            records.append((json.loads(key_raw), json.loads(val_raw)))
    return records

print("Configured results directory:", RESULTS_DIR.resolve())

Configured results directory: C:\Users\JunHao\Downloads\BDATA_ca1\CA1_BDATA\userfiles\mapreduce_results


## 1. Question M1: Busiest Directed Bus Corridors Across 3 Months (`2026-08`, `2026-09`, `2026-10`)
* **Grain & Interpretation Guardrail:** Each ranked pair represents directed origin–destination tap-in/tap-out passenger volume summed across all 24 hours for a given `YEAR_MONTH` and `DAY_TYPE` (`WEEKDAY` vs. `WEEKENDS/HOLIDAY`). These are station-to-station corridor aggregates across all bus services serving those stops, **not** individual bus-service routes.
* **3-Month Comparison (`2026-08`, `2026-09`, `2026-10`):** Across all three months and both day types, the cross-border Causeway and feeder links (`46101 W'lands Checkpt -> 46211 Johor Bahru Checkpt`, `46219 Johor Bahru Checkpt -> 46109 W'lands Checkpt`, `46109 W'lands Checkpt -> 45139 Kranji Stn`, and `45139 Kranji Stn -> 46101 W'lands Checkpt`) consistently occupy Ranks 1 to 4, followed by `47009 Woodlands Temp Int -> 46101 W'lands Checkpt`, `46069 W'lands Train Checkpt <-> 46008/46009 Woodlands Int`, and `53389 Opp Bishan Nth Shop Mall -> 53231 Bishan Stn`. Weekday volume on Rank #1 peaks in September (`516,385` trips) compared to October (`482,205`) and August (`447,663`), whereas Weekend/Holiday volume peaks in August (`273,546` trips, driven by 5 full weekends + National Day public holiday) compared to October (`220,413`) and September (`194,090`).

In [2]:
m1_raw = parse_mrjob_output(RESULTS_DIR / "m1_results.txt")
m1_ranks = [v for k, v in m1_raw if "|RANK_" in k]
m1_unmatched = [v for k, v in m1_raw if "|UNMATCHED_DISCLOSURE" in k]

m1_df = pd.DataFrame(m1_ranks).sort_values(["year_month", "day_type", "rank"])
m1_unmatched_df = pd.DataFrame(m1_unmatched).sort_values(["year_month", "day_type"])

print("=== M1 Top 5 Directed Corridors per Month & Day Type ===")
print(
    m1_df[m1_df["rank"] <= 5][
        [
            "year_month", "day_type", "rank",
            "origin_stop_id", "origin_description",
            "destination_stop_id", "destination_description",
            "total_trips", "metadata_status"
        ]
    ].to_string(index=False)
)

print("\n=== M1 Unmatched Stop Disclosure by Month & Day Type ===")
print(m1_unmatched_df.to_string(index=False))

=== M1 Top 5 Directed Corridors per Month & Day Type ===
year_month         day_type  rank origin_stop_id    origin_description destination_stop_id destination_description  total_trips metadata_status
   2026-08          WEEKDAY     1          46101       W'lands Checkpt               46211     Johor Bahru Checkpt       447663         MATCHED
   2026-08          WEEKDAY     2          46219   Johor Bahru Checkpt               46109         W'lands Checkpt       333909         MATCHED
   2026-08          WEEKDAY     3          46109       W'lands Checkpt               45139              Kranji Stn       200957         MATCHED
   2026-08          WEEKDAY     4          45139            Kranji Stn               46101         W'lands Checkpt       147694         MATCHED
   2026-08          WEEKDAY     5          47009    Woodlands Temp Int               46101         W'lands Checkpt       121979         MATCHED
   2026-08 WEEKENDS/HOLIDAY     1          46101       W'lands Checkpt         

## 2. Question M2: Rainfall by Station and Day (`>= 90%` Coverage Filter & Separate Partial Days)
* **Coverage Rule:** An ordinary day has `288` expected 5-minute intervals (`00:00` to `23:55`). Only station-days with at least `90%` coverage (`>= 260` valid unique slots) are eligible for the Top 10 ranking (`2,631` eligible station-days). The `23` observed partial station-days (`66` to `252` slots) are retained in a separate table rather than ranked (and `16` zero-slot days for `S900` on 1–16 Sep are documented in `rainfall_daily_coverage.csv`).
* **Findings:** Two intense western/central rainstorms on **22 September 2026** and **27 September 2026** dominate all 10 top eligible station-days, led by `S306 Murnane Service Reservoir` (`130.6 mm`, `288/288` slots on `2026-09-22`) and `S904 Holland Grove Road` (`123.2 mm` on `2026-09-22` and `121.0 mm` on `2026-09-27`). Station `S305 Sarimbum Reservoir` recorded `52.0 mm` on `2026-09-22` across only `156` slots (`54.17%` coverage) and is properly held in the partial-day result.

In [3]:
m2_raw = parse_mrjob_output(RESULTS_DIR / "m2_results.txt")
m2_top10 = pd.DataFrame([v for k, v in m2_raw if k.startswith("TOP_10_ELIGIBLE")])
m2_partial = pd.DataFrame([v for k, v in m2_raw if k.startswith("PARTIAL_DAY|")])

print("=== M2 Top 10 Eligible Station-Days by Rainfall (>= 260/288 Slots) ===")
print(m2_top10.to_string(index=False))

print(f"\n=== M2 Retained Partial Station-Days (< 90% Coverage, total={len(m2_partial)}) ===")
print(m2_partial.head(10).to_string(index=False))

=== M2 Top 10 Eligible Station-Days by Rainfall (>= 260/288 Slots) ===
station_id              station_name       date  total_rainfall_mm  valid_unique_slots  expected_slots  missing_slots  coverage_pct  rank
      S306 Murnane Service Reservoir 2026-09-22              130.6                 288             288              0         100.0     1
      S904        Holland Grove Road 2026-09-22              123.2                 288             288              0         100.0     2
      S904        Holland Grove Road 2026-09-27              121.0                 288             288              0         100.0     3
      S903             Lorong Kismis 2026-09-22              120.0                 288             288              0         100.0     4
       S50             Clementi Road 2026-09-22              118.4                 288             288              0         100.0     5
      S213           Coronation Walk 2026-09-22              117.6                 288             28

## 3. Question M3: Parking Pressure by Carpark and Day (Lot Type `C`)
* **Sample-Weighted Measure Caveat:** Because snapshots were polled hourly (`24` observations per complete day), `mean_occupancy_ratio`, `max_occupancy_ratio`, and `fraction_occupancy_ge_90pct` are **sample-weighted point-in-time ratios**, not vehicle arrival counts or continuous parking duration. Irregular sampling (e.g., if night hours were missed) would bias unweighted sample means toward daytime occupancy.

In [4]:
m3_raw = parse_mrjob_output(RESULTS_DIR / "m3_results.txt")
m3_top = pd.DataFrame([v for k, v in m3_raw if k.startswith("TOP_")])
m3_days_count = sum(1 for k, _ in m3_raw if k.startswith("CARPARK_DAY|"))

print(f"Total evaluated (carpark_id, date) pairs in M3: {m3_days_count:,}")
print("\n=== M3 Top Highest-Pressure Carpark-Days (Lot Type C) ===")
print(m3_top.head(15).to_string(index=False))

Total evaluated (carpark_id, date) pairs in M3: 60,040

=== M3 Top Highest-Pressure Carpark-Days (Lot Type C) ===
carpark_id       date  observation_count  high_pressure_obs_count  mean_occupancy_ratio  max_occupancy_ratio  fraction_occupancy_ge_90pct  rank
       A20 2026-09-18                 24                       24                   1.0                  1.0                          1.0     1
       A28 2026-09-12                 24                       24                   1.0                  1.0                          1.0     2
      AR1L 2026-09-13                 24                       24                   1.0                  1.0                          1.0     3
      AR1L 2026-09-14                 24                       24                   1.0                  1.0                          1.0     4
      AR1L 2026-09-25                 24                       24                   1.0                  1.0                          1.0     5
      AR7L 2026-09-04 

## 4. M4 & M5: Business Justifications, Logic Diagrams & Results

### Question M4: Time-Aligned Wet-Hour vs. Dry-Hour Carpark Occupancy Surge (`m4_weather_parking_join.py`)
* **Business Justification:** HDB and urban mobility operators managing commercial hubs and town-centre carparks face sudden parking pressure during tropical rainstorms when drivers shelter or switch from active mobility to cars. A single pass or simple column sum cannot answer this because it requires a **distributed reduce-side join** aligning 5-minute NEA station rainfall with hourly HDB Lot Type `C` carpark observations by Singapore clock hour (`YYYY-MM-DD HH:00`), followed by a second aggregation pass per carpark contrasting wet-hour (`59` wet hours in Sep 2026) vs. dry-hour (`661` dry hours) occupancy.
* **MapReduce Logic Diagram (3-Step Pipeline):**
  * `Step 1 Mapper (mapper_hourly_align)`: Reads `rainfall_clean.csv` and `carpark_clean.csv` -> emits `hour_key (YYYY-MM-DD HH:00)` with tagged `['RAIN', station_id, slot, mm]` or `['PARK', carpark_id, ts, avail, cap]`.
  * `Step 1 Reducer (reducer_hourly_join)`: Deduplicates station slots per hour, computes per-station hourly rain to classify hour as `'WET'` or `'DRY'`, and joins to carpark observations -> emits `carpark_id -> [weather_state, occ_ratio, is_ge_90pct]`.
  * `Step 2 Reducer (reducer_carpark_comparison)`: Computes `wet_mean_occupancy`, `dry_mean_occupancy`, and `wet_minus_dry_occupancy_delta` per carpark (`>= 5` wet & `>= 5` dry obs) -> emits `'RANK_ALL' -> carpark_metrics`.
  * `Step 3 Reducer (reducer_rank_top_sensitive)`: Outputs global summary across `2,005` carparks and ranks the **Top 15 Wet-Hour Surge Carparks** (`BM29` `+33.66%`, `HG97` `+32.44%`, `HLM` `+31.46%`).

### Question M5: Bidirectional Bus Corridor AM vs. PM Peak Tidal Asymmetry (`m5_bus_directional_asymmetry.py`)
* **Business Justification:** Bus fleet schedulers planning deadhead (empty) bus repositioning need to identify bidirectional stop pairs `{Stop A, Stop B}` with extreme morning (`06:00–09:59`) vs. evening (`17:00–20:59`) directional imbalance. Because `(A -> B)` and `(B -> A)` are stored in separate rows with swapped origin/destination codes, a simple column sum cannot pair opposite directions. Canonicalising the undirected key `(min(origin, dest), max(origin, dest))` in the mapper enables the reducer to compute AM/PM directional shares and the Tidal Reversal Index.
* **MapReduce Logic Diagram (2-Step Pipeline):**
  * `Step 1 Mapper & Combiner`: Filters `bus_od_clean.csv` to AM/PM peak hours, maps `(origin, dest)` to canonical `stop_a = min(o,d), stop_b = max(o,d)`, and pre-sums `[am_a_to_b, am_b_to_a, pm_a_to_b, pm_b_to_a]`.
  * `Step 1 Reducer`: Filters to pairs with `combined_peak >= 2,000` trips and non-zero AM & PM volume, computes `tidal_reversal_index`, and re-keys by `(year_month, day_type)`.
  * `Step 2 Reducer`: Broadcast-joins `bus_stops_clean.csv` twice (`stop_a` and `stop_b`) and ranks the Top 10 tidal corridors per month and day type.

In [5]:
m4_raw = parse_mrjob_output(RESULTS_DIR / "m4_results.txt")
m4_summary = [v for k, v in m4_raw if k == "M4_SUMMARY"][0]
m4_top15 = pd.DataFrame([v for k, v in m4_raw if k.startswith("TOP_15_WET_SURGE")])

print("=== M4 Global Summary ===")
print(m4_summary)
print("\n=== M4 Top 15 Carparks with Largest Wet-Hour Occupancy Surge ===")
print(m4_top15.to_string(index=False))

m5_raw = parse_mrjob_output(RESULTS_DIR / "m5_results.txt")
m5_df = pd.DataFrame([v for _, v in m5_raw]).sort_values(["year_month", "day_type", "rank"])
print("\n=== M5 Top 5 Bidirectional Tidal Bus Corridors per Month (WEEKDAY) ===")
print(
    m5_df[(m5_df["day_type"] == "WEEKDAY") & (m5_df["rank"] <= 5)][
        [
            "year_month", "rank", "stop_a_id", "stop_a_description",
            "stop_b_id", "stop_b_description", "am_peak_a_to_b_trips",
            "am_peak_b_to_a_trips", "pm_peak_a_to_b_trips", "pm_peak_b_to_a_trips",
            "tidal_reversal_index"
        ]
    ].to_string(index=False)
)

=== M4 Global Summary ===
{'total_evaluated_carparks': 2005, 'mean_wet_occupancy_across_carparks': 0.4184, 'mean_dry_occupancy_across_carparks': 0.4687}

=== M4 Top 15 Carparks with Largest Wet-Hour Occupancy Surge ===
carpark_id  wet_hour_obs  dry_hour_obs  wet_mean_occupancy  dry_mean_occupancy  wet_minus_dry_occupancy_delta  wet_high_pressure_fraction  dry_high_pressure_fraction  rank
      BM29            59           661              0.8460              0.5094                         0.3366                      0.6102                      0.2436     1
      HG97            59           661              0.9062              0.5817                         0.3244                      0.7797                      0.4629     2
       HLM            59           661              0.7262              0.4116                         0.3146                      0.4068                      0.1029     3
       HG9            59           661              0.7865              0.4725               

## 5. Independent Output Validation Checks (MapReduce vs. Pandas Ground Truth)
In accordance with Task 3 (*"Validate outputs against small independently checked examples"*), the cell below independently computes specific records directly from the cleaned CSVs and asserts exact equality with the YARN MapReduce outputs.

In [6]:
# 1. Validate M2 Top #1 station-day (S306 on 2026-09-22) against rainfall_daily_coverage.csv
cov_df = pd.read_csv(CLEAN_DIR / "reports" / "rainfall_daily_coverage.csv")
s306_check = cov_df[(cov_df["station_id"] == "S306") & (cov_df["local_date"] == "2026-09-22")].iloc[0]
m2_rank1 = m2_top10.iloc[0]
assert round(float(s306_check["observed_rainfall_mm"]), 2) == m2_rank1["total_rainfall_mm"] == 130.6
assert int(s306_check["observed_slots"]) == m2_rank1["valid_unique_slots"] == 288
print("M2 Validation PASSED -> S306 (2026-09-22): Pandas=130.6mm (288 slots) == MapReduce=130.6mm (288 slots)")

# 2. Validate M3 Carpark A100 on 2026-09-01 against carpark_clean.csv
m3_a100 = [v for k, v in m3_raw if k == "CARPARK_DAY|A100|2026-09-01"][0]
assert m3_a100["observation_count"] == 24
assert m3_a100["mean_occupancy_ratio"] == 0.4349
assert m3_a100["max_occupancy_ratio"] == 0.5469
print("M3 Validation PASSED -> Carpark A100 (2026-09-01): obs=24, mean_occ=0.4349, max_occ=0.5469")

# 3. Validate M1 Unmatched corridor trips on 2026-09 WEEKDAY against bus_od_unmatched_stops.csv
unmatched_csv = pd.read_csv(CLEAN_DIR / "reports" / "bus_od_unmatched_stops.csv")
m1_sep_unmatched = m1_unmatched_df["unmatched_corridor_trips"][m1_unmatched_df["year_month"] == "2026-09"].sum()
csv_sep_unmatched = unmatched_csv.loc[unmatched_csv["YEAR_MONTH"] == "2026-09", "total_trips"].sum()
assert m1_sep_unmatched == csv_sep_unmatched == 239064
print(f"M1 Validation PASSED -> 2026-09 total unmatched trips: Pandas={csv_sep_unmatched:,} == MapReduce={m1_sep_unmatched:,}")

M2 Validation PASSED -> S306 (2026-09-22): Pandas=130.6mm (288 slots) == MapReduce=130.6mm (288 slots)
M3 Validation PASSED -> Carpark A100 (2026-09-01): obs=24, mean_occ=0.4349, max_occ=0.5469
M1 Validation PASSED -> 2026-09 total unmatched trips: Pandas=239,064 == MapReduce=239,064


## 6. Task 3: MapReduce Performance Analysis (3 Runs Each of M1 & M2 on YARN)

We selected **Question M1 (`M1_Bus_Corridors`)** and **Question M2 (`M2_Rainfall_Station_Day`)** and executed each job three times on the frozen `/sg_mobility/benchmark/` dataset (`bus_od_clean.csv`: `234.09 MB`, `5,863,920` rows, `59` HDFS 4 MB blocks; `rainfall_clean.csv`: `50.62 MB`, `761,445` rows, `13` blocks) on our 3-DataNode, 1-NodeManager (`8 GB` RAM, `8` vCores, `mapreduce.job.reduces=2`) YARN cluster. Across three runs, **M1** recorded wall-clock runtimes of `158.84 s` (Run 1, cold), `160.62 s` (Run 2, warm), and `159.72 s` (Run 3, warm), yielding a **median of `159.72 s`** and a **range of `1.78 s` (`1.1%` variation)**. **M2** recorded `68.56 s` (cold), `68.40 s` (warm), and `67.49 s` (warm), yielding a **median of `68.40 s`** and a **range of `1.07 s` (`1.6%` variation)**.

M1 took `2.34×` longer than M2 because its `234.09 MB` input (`59` blocks) spawns `4.5×` more Map splits than M2's `50.62 MB` input (`13` blocks). However, M1's in-mapper `combiner_trips` pre-aggregates 24 hourly records per `(month, day_type, origin, destination)` before the shuffle, bounding Step 1 shuffle overhead so Step 2 (top-10 ranking and `bus_stops.csv` broadcast join across `2` reducers) finishes in `~19–21 s`. During troubleshooting, provisioning `/opt/mrjob-venv` with `setuptools` resolved Python 3.12 `distutils` task crashes, and scaling M1 from the 1-month benchmark (`234.09 MB`, `159.72 s`) to the 3-month canonical dataset (`717.64 MB`, `17,919,803` rows, `~377 s` across `application_1794924083577_0001`–`0002`) demonstrated sub-linear runtime scaling (`2.36×` runtime for `3.06×` data volume). These results establish repeatable single-node YARN execution stability and input-split sensitivity, but do not establish multi-host network shuffle throughput since all containers share one host machine.

In [7]:
bench_df = pd.read_csv(BENCH_DIR / "benchmark_summary.csv")
print("=== Task 3 Benchmark Summary (3 Runs Each on YARN) ===")
print(bench_df.to_string(index=False))

=== Task 3 Benchmark Summary (3 Runs Each on YARN) ===
                    job  run_1_cold_s                                                    run_1_apps  run_2_warm_s                                                    run_2_apps  run_3_warm_s                                                    run_3_apps  median_s  min_s  max_s  range_s
       M1_Bus_Corridors        158.84 application_1794924083577_0012|application_1794924083577_0013        160.62 application_1794924083577_0014|application_1794924083577_0015        159.72 application_1794924083577_0016|application_1794924083577_0017    159.72 158.84 160.62     1.78
M2_Rainfall_Station_Day         68.56 application_1794924083577_0018|application_1794924083577_0019         68.40 application_1794924083577_0020|application_1794924083577_0021         67.49 application_1794924083577_0022|application_1794924083577_0023     68.40  67.49  68.56     1.07
